# 🧩 토크나이제이션: 텍스트를 LLM이 이해하는 언어로 변환하기

> **핵심 질문:** LLM은 텍스트가 아닌 **숫자만 이해**합니다. 그렇다면 "machine learning"이라는 단어를 어떻게 숫자로 변환할까요? 그리고 "machines", "learning", "learnings" 같은 변형된 단어들은 어떻게 처리할까요?

---

## 📌 배경: 왜 토크나이제이션이 어려운가?

### 1. 사람 vs LLM의 관점 차이

#### 사람의 이해
```
"machine learning"을 읽음
      ↓
의미적으로 "기계학습"이라고 직관적 이해
      ↓
관련 정보 연상 (neural networks, AI, etc.)
```

#### LLM의 처리
```
"machine learning"을 입력받음
      ↓
"machine"과 "learning" → 각각을 숫자로 변환 필요
      ↓
"machine" → [1, 2, 3, 4, 5, 6, 7]?  (문자별)
          → [4572]?  (단어별)
          → [456, 78]?  (서브워드별)
      ↓
어떤 방식이 최적일까?
```

### 2. 3가지 선택지와 각각의 문제점

#### 선택지 1: 문자 단위 처리

```
"machine" → ['m', 'a', 'c', 'h', 'i', 'n', 'e']
           → [1, 2, 3, 4, 5, 6, 7]  (7개 토큰)
```

**문제점:**
- 📊 **정보 분산**: 하나의 의미("machine")가 7개로 쪼개짐
- 🔄 **시퀀스 길이 증가**: 문장이 길어지면 처리할 토큰이 7배 증가
- 💾 **메모리/계산량 증가**: Attention 계산이 O(n²) → 7배 시퀀스면 49배 계산량!
- ❌ **문맥 손실**: 'm', 'a', 'c'가 "machine"의 의미를 담지 못함

**예: 한글 처리**
```
"안녕하세요" (5글자) → ['안', '녕', '하', '세', '요'] (5개 토큰)
문제: 각 글자가 의미를 담지 못함
```

#### 선택지 2: 단어 단위 처리

```
"machine" → ['machine'] → [4572]
"machines" → ['machines'] → [9999]  ← 새로운 ID 필요!
"machinery" → ['machinery'] → [8888]  ← 또 새로운 ID!
```

**문제점:**
- 📚 **어휘 폭발(Vocabulary Explosion)**: 단어의 모든 변형을 학습해야 함
  - run, runs, running, ran, runner, ...
  - 영어: 약 100만 단어 이상
  - 메모리 폭증, 학습 비효율

- ❌ **OOV (Out-of-Vocabulary) 문제**: 학습하지 않은 단어는 처리 불가
  - "ChatGPT"라는 신조어가 학습 데이터에 없으면?
  - 미지의 단어에 대해 모델이 답할 수 없음

- 🌍 **언어별 복잡성**: 한글의 띄어쓰기 모호성
```
"학습을 합니다" vs "학습을합니다"
단어 경계가 명확하지 않음
```

#### 선택지 3: 서브워드 토크나이제이션 ✨

```
"machine" → ['machine']  (1개 토큰, 의미 보존)
"machines" → ['machine', '##s']  (2개 토큰, 변형 처리)
"machinery" → ['machine', '##ry']  (2개 토큰, 자동 조합)

장점:
✓ 의미 보존 ("machine"이 명시적)
✓ 신조어 처리 가능 (기본 블록으로 조합)
✓ 메모리 효율 (어휘 수 제한)
✓ 언어 보편성 (모든 언어에 적용)
```

### 3. 핵심 개념: 글자당 토큰 소모율

**토크나이제이션 방식에 따라 같은 정보를 표현하는 비용이 완전히 다릅니다.**

#### 예시: 같은 의미를 다양한 방식으로 표현

```
의미: "안녕하세요" (인사말 1개)

1️⃣ 문자 단위:
   ['안', '녕', '하', '세', '요']
   → 5개 토큰
   → 글자당 토큰: 5개 ÷ 5글자 = 1.00개/글자

2️⃣ 단어 단위:
   ['안녕하세요']
   → 1개 토큰
   → 글자당 토큰: 1개 ÷ 5글자 = 0.20개/글자

3️⃣ 최적 서브워드:
   ['안녕', '하세요']
   → 2개 토큰
   → 글자당 토큰: 2개 ÷ 5글자 = 0.40개/글자
```

**비용 비교 (API 요금 기준):**
```
1,000글자 텍스트 처리시:

문자 단위:     1,000 토큰 = $0.50 ❌❌❌
서브워드:       400 토큰 = $0.20 ✓
단어 단위 (이상적): 200 토큰 = $0.10 (현실적으로 불가능)

→ 서브워드가 문자 단위의 절반 비용!
```

### 4. 실제 언어별 비용 차이

**같은 내용이지만, 언어에 따라 비용이 3배까지 차이납니다!**

#### 실제 예시

| 언어 | 예시 | 글자수 | 토큰수 | 글자당 소모 | API 비용 |
|------|------|--------|--------|-----------|----------|
| **영어** | "hello world" | 11 | 2 | **0.18** | **1배** (기준) |
| **한글** | "안녕하세요" | 5 | 2 | **0.40** | **2.2배** 비쌈 |
| **중국어** | "你好" | 2 | 2 | **1.00** | **5.5배** 비쌈 |

#### 실무 영향 (1,000글자 처리 기준)

```
영어:   1,000 글자 × 0.18 = 약 180 토큰   → $0.09
한글:   1,000 글자 × 0.40 = 약 400 토큰   → $0.20 (영어의 2.2배)
중국어: 1,000 글자 × 1.00 = 약 1,000 토큰 → $0.50 (영어의 5.5배!)
```

**왜 이런 차이가 날까?**
1. 영어: 단어들이 길고 명확한 경계 (공백) 존재
2. 한글: 조사/어미 결합으로 단어가 길어짐 + 문법 변화 복잡
3. 중국어: 각 한자마다 어휘적/문법적 의미 (띄어쓰기 없음)

---

## 🔍 핵심 개념: 적절한 크기의 단위 찾기

### 토크나이제이션의 목표

**"의미를 최대한 보존하면서, 비용을 최소화하는 단위 크기 찾기"**

```
← 너무 작음                최적                너무 큼 →
[문자]    →    [서브워드]    →    [단어]

비용:   높음      중간       낮음(비현실)
의미:   분산      보존       집중
유연성: 높음      중간       낮음(OOV)
```

### 균형 찾기

| 측면 | 문자 | 서브워드 ✓ | 단어 |
|------|------|-----------|------|
| 토큰 수 | 많음 | 중간 | 적음 |
| 메모리 | 적음 ✓ | 중간 ✓ | 많음 |
| 의미 보존 | 낮음 | 높음 ✓ | 높음 |
| OOV 처리 | 우수 ✓ | 우수 ✓ | 취약 |
| 계산 속도 | 느림 | 중간 ✓ | 빠름 |

---

## 💡 3가지 토크나이제이션 방식

### 1️⃣ BPE (Byte Pair Encoding) - 빈도 기반

**사용 모델:** GPT-2, GPT-3, GPT-4, Llama

**동작 원리:**
1. 모든 단어를 문자 단위로 분할
2. 가장 자주 인접하는 문자 쌍을 찾아 병합
3. 반복하여 어휘 크기에 도달

**예시:**
```
Step 1: ['l', 'o', 'w', 'e', 'r']
Step 2: 'e'와 'r'이 자주 나옴 → 'er' 병합
        ['l', 'o', 'w', 'er']
Step 3: 'l'과 'o'가 자주 나옴 → 'lo' 병합
        ['lo', 'w', 'er']
Step 4: 'lo'와 'w'가 자주 나옴 → 'low' 병합
        ['low', 'er']
```

**장점:**
- ✓ 신조어도 기본 문자로 표현 가능
- ✓ 어휘 크기 조절 가능

**단점:**
- ❌ 의미와 무관하게 빈도만 고려
- ❌ **한글 처리 극악**: 자음/모음까지 분리

**한글 문제:**
```
"기계 학습을 좋아합니다"
↓ (BPE 처리)
['기', '계', '학', '습', '을', '좋', '아', '합', '니', '다']
→ 10개 토큰 (비효율!)
→ 자음/모음 분리로 의미 완전 손실
```

### 2️⃣ WordPiece - 확률 기반

**사용 모델:** BERT, RoBERTa, KoBERT

**동작 원리:**
1. 미리 정의된 어휘 사전 사용
2. 확률이 **가장 높은** 단어 조합 선택
3. 단어 내부 서브워드는 `##` 접두사로 표시

**예시:**
```
Input: "playing"
Output: ['play', '##ing']
        (어근 'play' + 어미 'ing'를 명시적으로 표시)
```

**장점:**
- ✓ 의미 기반 분할 (확률 최대화)
- ✓ 문법 변화 자동 처리
- ✓ 중간 정도의 한글 처리

**단점:**
- ❌ 사전이 고정되어 신조어 추가 어려움
- ❌ 띄어쓰기 의존 (한글 처리 부분적)

### 3️⃣ SentencePiece - 언어 독립적

**사용 모델:** mBART, XLM-RoBERTa, Llama 2

**동작 원리:**
1. 모든 언어를 동일하게 처리 (공백 포함)
2. 공백을 메타스페이스 `▁`로 표시
3. BPE와 유사하지만 언어 무관적

**예시:**
```
Input (영어):   "get up"
변환:           "get▁up"
Output:         ['get', '▁up']

Input (한글):   "일어나"
변환:           "▁일어나"
Output:         ['▁일', '어나']  또는  ['▁일어나']
```

**장점:**
- ✓ 진정한 다국어 지원 (모든 언어 동등 처리)
- ✓ 공백 정보 보존
- ✓ **한글 처리 최고** (의미 보존 + 효율)
- ✓ 새로운 언어도 자동 처리

**단점:**
- ❌ 토큰 수가 조금 많을 수 있음
- ❌ 처리 속도 약간 느림

---

## 📊 3가지 방식 상세 비교

| 항목 | BPE | WordPiece | SentencePiece |
|------|-----|-----------|---------------|
| **기반** | 문자 빈도 | 확률 기반 | 바이트 레벨 |
| **알고리즘** | Greedy 병합 | 확률 최대화 | BPE 변형 |
| **대표 모델** | GPT 계열 | BERT 계열 | 다국어 모델 |
| **어휘 크기** | 50K | 30K | 32K~64K |
| **한글 처리** | ⭐ (최악) | ⭐⭐ (중간) | ⭐⭐⭐ (최고) |
| **영어 효율** | ⭐⭐⭐ | ⭐⭐⭐ | ⭐⭐⭐ |
| **다국어** | ⭐ (약함) | ⭐⭐ (중간) | ⭐⭐⭐ (우수) |
| **의미 보존** | ⭐⭐ | ⭐⭐⭐ | ⭐⭐ |
| **신조어 처리** | ⭐⭐⭐ | ⭐⭐ | ⭐⭐⭐ |
| **공백 처리** | Ġ | [없음] | ▁ |
| **사용 가능성** | 매우 높음 | 높음 | 점점 높아짐 |

---

## 🎯 학습 체크리스트

### 이 단계를 완료했다면 다음을 이해해야 합니다:

- [ ] **LLM이 왜 숫자만 이해하는가**
- [ ] **문자/단어 단위의 한계**
- [ ] **글자당 토큰 소모율 개념**
- [ ] **같은 정보도 언어에 따라 비용이 3배까지 다른 이유**
- [ ] **BPE가 한글에서 극악인 이유** (자음/모음 분리)
- [ ] **WordPiece의 장점** (##마크, 의미 기반)
- [ ] **SentencePiece가 다국어 최고인 이유** (언어 독립)
- [ ] **각 방식의 트레이드오프 이해**

### 다음 단계

**02_practice.ipynb**에서 이 3가지 방식을 **실제로 코드로 비교**해봅니다:
- GPT-2 (BPE) 토크나이저 실행
- mBERT (WordPiece) 토크나이저 실행
- XLM-RoBERTa (SentencePiece) 토크나이저 실행
- 같은 텍스트로 비교 분석
- 시각화로 패턴 확인